## Create pre-defined cross validation split

In [ ]:
# import json
# from utils.util import read_dataset, create_fold

# # read data
# data = "benbow" 
# filename = f"dataset/train_data/{data}.fasta" 

# X, y, groups, genera, species = read_dataset(filename)

# n_splits = 5
# n_repeats = 5
# all_runs = create_fold(X, y, species, n_splits=n_splits, n_repeats=n_repeats)

# # Save to file
# with open(f"dataset/folds/{data}_stratified_group_folds_new.json", "w") as f:
#     json.dump(all_runs, f, indent=2)

## Read the pre-defined cross validation split

In [1]:
from utils.util import read_fold 
data = "benbow"
labels = read_fold(f"dataset/folds/{data}_stratified_group_folds_new.json")

In [2]:
print(len(labels['fold_0']['train_idx']))
print(len(labels['fold_0']['valid_idx']))
print(len(labels['fold_0']['test_idx']))

1538
539
492


## Run predict.py to cross-validate different data representations and models

In [ ]:
#run python run_crossval.py --n-worker 6 --representations-index 1 
#through run_crossval_parallel.sh 

## Read cross-validation results

In [4]:
import os 
import numpy as np
import pandas as pd 
import altair as alt 
from utils.util import read_results, compute_metrics, read_dataset, read_fold 

In [5]:
# define the data set (benbow, rvm, gicluster)
train_dataname = "benbow" 

# Read training data
train_filename = f"dataset/train_data/{train_dataname}.fasta"
X_train, y_train, groups_train, genera_train, species_train = read_dataset(train_filename)

# Read fold
fold_filename = f"dataset/folds/{train_dataname}_stratified_group_folds_new.json"
labels = read_fold(fold_filename)

mypath = "predictions/predictions_train_val_test"
files = [f for f in os.listdir(mypath) if os.path.isfile(os.path.join(mypath, f))]

predictions_df = pd.DataFrame()

for file in files:
    if file.endswith('.xlsx') and file.startswith(f'{train_dataname}_predictions'):
        filename = os.path.join(mypath,file)
        df = read_results(filename, header=['fold','representation','model'])
        predictions_df = pd.concat([predictions_df,df])

# Filter out models that are not of interest (e.g., ensemble methods)
predictions_df = predictions_df[~predictions_df['model'].isin(['AdaBoost','GradientBoosting','XGBoost','Bagging'])]

# Create a label column to identify each (feature_a, feature_b) pair
predictions_df['pair_id'] = predictions_df['representation'].astype(str) + '/' + predictions_df['model'].astype(str)

eval_metrics = compute_metrics(predictions_df, labels, y_train)
eval_metrics['representation'] = eval_metrics['pair_id'].apply(lambda x: x.split('/')[0])
eval_metrics['model'] = eval_metrics['pair_id'].apply(lambda x: x.split('/')[1])

In [6]:
overall_metrics = eval_metrics.groupby(['model','representation']).agg({'f1':'mean','accuracy':'mean','precision':'mean','recall':'mean','mcc':'mean'}).reset_index().sort_values(by=['f1'], ascending=False)
overall_metrics = overall_metrics.replace({'LogisticRegression':1, 'NaiveBayes':2, 'DecisionTree':3, 'RandomForest':4, 'SVM':5})
overall_metrics['rank'] = overall_metrics.groupby('representation')['f1'].rank(method="first", ascending=False)

# Step 1: Melt the DataFrame
metrics = ['accuracy', 'precision', 'recall', 'f1', 'mcc']
df_long = overall_metrics.melt(
    id_vars=['model', 'representation'],
    value_vars=metrics,
    var_name='metric',
    value_name='value'
)

# Step 2: Create the Altair chart
# Dropdown selector
metric_param = alt.param(
    bind=alt.binding_select(options=metrics, name='Metric:'),
    value='f1'
)

# Heatmap
overall_heatmap = alt.Chart(df_long).mark_rect().encode(
    x=alt.X('representation:N', title='Representation', axis=None),#axis=alt.Axis(labelAngle=45)),
    y=alt.Y('model:N', title='Model'),
    color=alt.Color('value:Q', scale=alt.Scale(scheme='greys'), legend=alt.Legend(title='F1 score', orient='top',)),
    tooltip=['metric', 'value']
).transform_filter(
    (alt.datum.metric == metric_param) 
).add_params(
    metric_param,
).properties(
    title=alt.TitleParams('a)', anchor='end', orient='left', angle=0)  # Add Left-side title
)


# Heatmap with highlighted box for top model for each representation
highlight_condition = alt.condition(
    alt.datum.rank == 1,  # condition
    alt.value('black'),    # stroke color if True
    alt.value('white')     # stroke color if False
)

selected_heatmap = alt.Chart(overall_metrics).mark_rect(
    stroke='black',     # border color
    strokeWidth=1       # border thickness
).encode(
    x=alt.X('representation:N', title='Representation', axis=alt.Axis(labelAngle=45)),
    y=alt.Y('model:N', title='Model'),
    color=alt.condition(
        alt.datum.rank == 1,
        alt.value('black'),
        alt.value('white')
    )
).properties(
    title=alt.TitleParams('b)', anchor='end', orient='left', angle=0)  # Add Left-side title
)


fontsize=16
combined_chart = alt.vconcat(
    overall_heatmap, selected_heatmap
).resolve_scale(
    x='shared'  # or 'shared' if you want exact same y-scale
).configure_axis(
    labelFontSize=fontsize,
    titleFontSize=fontsize,
).configure_header(
    labelFontSize=fontsize,
    titleFontSize=fontsize,
)

combined_chart

/var/folders/ws/qg8ztb9n7yj1s15m1x9c1vdw0000gp/T/ipykernel_14039/3510840090.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  overall_metrics = overall_metrics.replace({'LogisticRegression':1, 'NaiveBayes':2, 'DecisionTree':3, 'RandomForest':4, 'SVM':5})


alt.VConcatChart(...)

## An example of kappa-error diagram

In [7]:
#import libraries
from itertools import combinations
import os 
from utils.util import kappa, pareto_n
from scipy.spatial import ConvexHull
from sklearn.metrics import f1_score, matthews_corrcoef

alt.data_transformers.disable_max_rows()
pd.set_option('display.max_rows', 500)

In [8]:
# define a function to compute pareto frontier and convex hull solutions, given kappa-error data
def compute_pfront_chull(data, obj_1, obj_2):
    # Create a DataFrame for the points
    df_points = data.copy()
    P = pareto_n(-df_points[[obj_1, obj_2]].values)

    indices = list(df_points.iloc[P[0], :].sort_values("kappa").index)

    df_points["pfront"] = -1
    df_points.iloc[indices, df_points.columns.get_loc("pfront")] = range(len(indices))

    hull = ConvexHull(df_points[[obj_1, obj_2]])

    df_points["chull_complete"] = -1
    df_points.iloc[hull.vertices, df_points.columns.get_loc("chull_complete")] = \
        range(hull.vertices.shape[0])

    #further reduce the convex hull to points towards lower, left corner
    df_hull = df_points.loc[df_points.chull_complete != -1, [obj_1,obj_2]]

    # mask convex hull (use only vals towards lower, left corner)
    P = pareto_n(-df_hull.values)

    indices = list(df_hull.iloc[P[0], :].sort_values(obj_1).index)

    df_points["chull"] = -1
    df_points.iloc[indices, df_points.columns.get_loc("chull")] = range(len(indices))

    return df_points

# define a function to return chart of kappa-error diagram
def draw_kappa_error_diagram(data, obj_1, obj_2):
    # Precompute a condition label
    def assign_color(row):
        if row['chull'] != -1: 
            return 'chull'
        elif row['pfront'] != -1:
            return 'pfront'
        else:
            return 'all'

    data['color_label'] = data.apply(assign_color, axis=1)

    # Define interactive point selection (replaces selection_multi)
    selection = alt.selection_point(fields=['color_label'], bind='legend', toggle=True)

    # Color mapping (optional: custom scale)
    color_scale = alt.Scale(domain=['chull', 'pfront', 'all'],
                            #range=['#e41a1c', '#377eb8', 'grey']
                            range=['#1b9e77','#7570b3','lightgrey'])

    # Create the scatter plot
    scatter = alt.Chart(data).mark_circle(size=30, stroke='black', strokeWidth=0.5).encode( 
        x=alt.X(obj_1, axis=alt.Axis(values=list(np.arange(0,1.1,0.1))), title=obj_1),
        y=alt.Y(obj_2, axis=alt.Axis(values=[0, 0.1, 0.2, 0.3, 0.4]), title=obj_2),
        tooltip=['pair_id', obj_1, obj_2, 'Adjusted_RV'],
        color=alt.Color('color_label:N', scale=color_scale, legend=alt.Legend(title='Solution', orient='top',)),
        #opacity=alt.value(0.3)
        opacity=alt.condition(alt.datum.color_label=='all', alt.value(0.3), alt.value(1.0)),
    ).properties(
        width=800,
        height=600,
    )

    pfront_data = data[data['pfront'] != -1].copy()

    # Create the dash line 
    pfront_line = alt.Chart(pfront_data).mark_line(point=False, strokeDash=[5,5]).encode(
        x=alt.X(obj_1, axis=alt.Axis(values=list(np.arange(0,1.1,0.1))), title=obj_1),
        y=alt.Y(obj_2, axis=alt.Axis(values=[0, 0.1, 0.2, 0.3, 0.4]), title=obj_2),
        tooltip=['pair_id', obj_1, obj_2, 'Adjusted_RV'],
        color=alt.Color('color_label:N', scale=color_scale,legend=alt.Legend(title='Solution', orient='top',)),# legend=None),
    )

    chull_data = data[data['chull'] != -1].copy()

    # Create the dash line 
    chull_line = alt.Chart(chull_data).mark_line(point=False, strokeDash=[5,5]).encode(
        x=alt.X(obj_1, axis=alt.Axis(values=list(np.arange(0,1.1,0.1))), title=obj_1),
        y=alt.Y(obj_2, axis=alt.Axis(values=[0, 0.1, 0.2, 0.3, 0.4]), title=obj_2),
        tooltip=['pair_id', obj_1, obj_2, 'Adjusted_RV'],
        color=alt.Color('color_label:N', scale=color_scale,legend=alt.Legend(title='Solution', orient='top',)),# legend=None),
    )

    # Combine the scatter plot and the Pareto front line
    chart = scatter + pfront_line + chull_line
    chart = chart.configure_axis(
        labelFontSize=12,
        titleFontSize=14,
    )

    return chart

In [9]:
# compute error
error_data = []
pairwise_kappa = []
y = y_train

for fold in labels.keys():
    fold_predictions_df = predictions_df[predictions_df['fold']==fold]
    fold_predictions_df = fold_predictions_df.dropna(axis=1)
    pred_cols = [col for col in fold_predictions_df.columns if col.startswith('y_')]
    df_preds = fold_predictions_df.set_index('pair_id')[pred_cols]

    test_indices = labels[fold]['valid_idx']
    ground_truth = y[test_indices]

    for pair_id, row in df_preds.iterrows():
        pred_prob = row.values
        pred_prob = pred_prob[:len(test_indices)]
        # Apply custom threshold
        threshold = 0.5
        preds = ((1-pred_prob) >= threshold).astype(int)
        mcc = 1 - matthews_corrcoef(ground_truth, preds)
        f1 = 1 - f1_score(ground_truth, preds)

        if 'ground_truth' not in pair_id:
            error_data.append({
                'fold': fold,
                'pair_id': pair_id,
                'mcc_error_rate': mcc,
                'f1_error_rate': f1,
            })

    pair_ids = fold_predictions_df['pair_id'].unique()
    unique_pairs = list(combinations(pair_ids, 2))

    for (pair1, pair2) in unique_pairs:
        row1 = df_preds.loc[pair1].values[:len(test_indices)]
        row2 = df_preds.loc[pair2].values[:len(test_indices)]
        threshold = 0.5
        pred1 = ((1-row1) >= threshold).astype(int)
        pred2 = ((1-row2) >= threshold).astype(int)

        kappa_score = kappa(pred1, pred2)
        
        pairwise_kappa.append({
            'fold': fold,
            'pair_1': pair1,
            'pair_2': pair2,
            'kappa': kappa_score
        })

error_df = pd.DataFrame(error_data)
pairwise_kappa_df = pd.DataFrame(pairwise_kappa)
pairwise_kappa_df = pairwise_kappa_df.replace(np.nan,0)
pairwise_kappa_w_error_df = pd.merge(pairwise_kappa_df, error_df, left_on=['fold','pair_1'], right_on=['fold','pair_id'], how='left',suffixes=('_1', '_2'))
pairwise_kappa_w_error_df = pd.merge(pairwise_kappa_w_error_df, error_df, left_on=['fold','pair_2'], right_on=['fold','pair_id'], how='left',suffixes=('_1', '_2'))
pairwise_kappa_w_error_df['avg_mcc_error_rate'] = (pairwise_kappa_w_error_df['mcc_error_rate_1'] + pairwise_kappa_w_error_df['mcc_error_rate_2']) / 2
pairwise_kappa_w_error_df['avg_f1_error_rate'] = (pairwise_kappa_w_error_df['f1_error_rate_1'] + pairwise_kappa_w_error_df['f1_error_rate_2']) / 2
pairwise_kappa_w_error_df['pair_id'] = pairwise_kappa_w_error_df['pair_1'] + '&' + pairwise_kappa_w_error_df['pair_2']

In [10]:
error_df_reset = error_df.groupby('pair_id')[['f1_error_rate','mcc_error_rate']].mean().reset_index().sort_values('f1_error_rate', ascending=True)
error_df_reset['representation'] = error_df_reset['pair_id'].apply(lambda x: x.split('/')[0])
error_df_reset['model'] = error_df_reset['pair_id'].apply(lambda x: x.split('/')[1])
error_df_reset = error_df_reset[['representation', 'model', 'mcc_error_rate', 'f1_error_rate']]
#select best model for each representation with lowest error rate
best_models = error_df_reset.groupby('representation').apply(lambda x: x.nsmallest(1, 'f1_error_rate')).reset_index(drop=True)
best_models['pair_id'] = best_models['representation'] + '/' + best_models['model']

# filter pairwise_kappa_w_error_df to only include best models on pair_1 and pair_2
pairwise_kappa_w_error_df_best = pairwise_kappa_w_error_df[pairwise_kappa_w_error_df['pair_1'].isin(best_models['pair_id']) & pairwise_kappa_w_error_df['pair_2'].isin(best_models['pair_id'])]
pairwise_kappa_w_error_df_best = pairwise_kappa_w_error_df_best[['fold', 'pair_id', 'pair_1', 'pair_2', 'kappa', 'mcc_error_rate_1', 'mcc_error_rate_2', 'f1_error_rate_1', 'f1_error_rate_2', 'avg_mcc_error_rate', 'avg_f1_error_rate']]
#pairwise_kappa_w_error_df_best

/var/folders/ws/qg8ztb9n7yj1s15m1x9c1vdw0000gp/T/ipykernel_14039/3058617696.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  best_models = error_df_reset.groupby('representation').apply(lambda x: x.nsmallest(1, 'f1_error_rate')).reset_index(drop=True)


### Assign Adjusted RV correlation between representations

In [11]:
#pairwise_kappa_w_error_df_best = pairwise_kappa_w_error_df.copy()
adjusted_rv = pd.read_excel('outputs/dataset_correlation/adjusted_rv_benbow.xlsx')
reversed_pairs = adjusted_rv.copy()
reversed_pairs['Representation_1'], reversed_pairs['Representation_2'] = reversed_pairs['Representation_2'], reversed_pairs['Representation_1']
adjusted_rv = pd.concat([adjusted_rv, reversed_pairs], ignore_index=True)

# assign rv coefficient to kappa df
pairwise_kappa_w_error_df_best['Representation_1'] = pairwise_kappa_w_error_df_best['pair_1'].apply(lambda x: x.split('/')[0])
pairwise_kappa_w_error_df_best['Representation_2'] = pairwise_kappa_w_error_df_best['pair_2'].apply(lambda x: x.split('/')[0])
pairwise_kappa_w_error_df_best = pd.merge(pairwise_kappa_w_error_df_best, adjusted_rv, 
                                            left_on=['Representation_1', 'Representation_2'], 
                                            right_on=['Representation_1', 'Representation_2'], 
                                            how='left')
pairwise_kappa_w_error_df_best.fillna(1, inplace=True)

pairwise_kappa_w_error_df_best = pairwise_kappa_w_error_df_best[['fold','pair_id','pair_1','pair_2','kappa','mcc_error_rate_1',
                                                        'f1_error_rate_1','mcc_error_rate_2','f1_error_rate_2',
                                                        'avg_mcc_error_rate','avg_f1_error_rate','Adjusted_RV']]

### Select candidate classifiers per fold by applying multi-objective optimization such as pareto frontier, convex hull

In [ ]:
solutions_df = pd.DataFrame()

for fold in labels.keys():
    error_metric = 'f1'
    data = pairwise_kappa_w_error_df_best[pairwise_kappa_w_error_df_best['fold']==fold].reset_index(drop=True)
    obj_1 = 'kappa' # mean_kappa or kappa
    obj_2 = f'avg_{error_metric}_error_rate'

    solutions_per_fold_df = compute_pfront_chull(data, obj_1, obj_2)

    solutions_df = pd.concat([solutions_df, solutions_per_fold_df])

### Save solutions per fold

In [ ]:
dataname = train_dataname

multi_objective_dir = f'multi_objective_solutions/{dataname}_multi_objective_solutions'
# check if directory exists, if not create it
if not os.path.exists(multi_objective_dir):
    os.makedirs(multi_objective_dir)

    # assign dataname to solutions_df
    solutions_df['dataname'] = dataname
    # save solutions
    for fold in labels.keys():
        solutions_df[solutions_df['fold']==fold].to_excel(f'{multi_objective_dir}/{dataname}_multi_objective_solutions_{fold}.xlsx',index=False)
else:
    print(f"Directory {multi_objective_dir} already exists. Skipping saving solutions.")

### Read kappa-error data and multi-objective solutions by pareto frontiers and convex hull

In [12]:
dataname = "benbow"
# get solutions from pareto frontier and convex hull
solutions_df = pd.DataFrame()

# read solutions from files
for fold in labels.keys():
    solutions_per_fold_df = pd.read_excel(f'multi_objective_solutions/{dataname}_multi_objective_solutions/{dataname}_multi_objective_solutions_{fold}.xlsx')

    col = ['fold','pair_id','pair_1','pair_2','mcc_error_rate_1',
            'f1_error_rate_1','mcc_error_rate_2','f1_error_rate_2',
            'avg_mcc_error_rate','avg_f1_error_rate','Adjusted_RV','kappa']
    
    solutions_df = pd.concat([solutions_df, solutions_per_fold_df])


In [13]:
data = solutions_df[solutions_df['fold']=='fold_0'].copy()  # Use the first fold for visualization
error_metric = 'f1'  # or 'f1'
obj_1 = 'kappa' # mean_kappa or kappa
obj_2 = f'avg_{error_metric}_error_rate'
per_fold_chart = draw_kappa_error_diagram(data, obj_1, obj_2)
per_fold_chart#.save('figures/per_fold_kappa_error_chart.pdf') 

alt.LayerChart(...)

## Run boundaries predictions

In [14]:
# python run_predictGI.py 

## Read ensemble selection results

In [16]:
ensemble_selection_res = pd.read_excel("outputs/multi_objective_validation/ensemble_selection_benbow.xlsx")

In [17]:
import ast 

ensemble_selection_res['upper'] = ensemble_selection_res['best_fold_scores'].apply(lambda x: max(ast.literal_eval(x)))
ensemble_selection_res['lower'] = ensemble_selection_res['best_fold_scores'].apply(lambda x: min(ast.literal_eval(x)))
ensemble_selection_res['best_fold_scores'] = ensemble_selection_res['best_fold_scores'].apply(lambda x: ast.literal_eval(x))
ensemble_selection_res['n_classifier'] = ensemble_selection_res['candidates'].apply(lambda x: len(ast.literal_eval(x)))

In [18]:
fontsize = 18 

# Extract the best fold scores for each ensemble type
best_fold_scores = []
#store box plot
best_scores_df = pd.DataFrame()
all_plots = {}
for ensemble_method in ensemble_selection_res['ensemble_method'].unique():
    ensemble_selection_results = ensemble_selection_res[ensemble_selection_res['ensemble_method']==ensemble_method]
    box_plots = {}
    for ensemble_solution in ensemble_selection_results['solution'].unique():
        best_fold_scores = ensemble_selection_results[ensemble_selection_results['solution']==ensemble_solution]['best_fold_scores']
        best_scores = ensemble_selection_results[ensemble_selection_results['solution']==ensemble_solution]['best_mean_score']

        # Sample 2D array: 3 columns (e.g., 3 categories), 10 rows
        data = np.array([score for score in best_fold_scores.values]).T

        # Convert to DataFrame with column names
        df = pd.DataFrame(data, columns=range(1,data.shape[1]+1))

        # Melt the DataFrame into long format
        df_long = df.melt(var_name='n_classifiers', value_name='score')

        # Create the box (whisker) plot
        box_plot = alt.Chart(df_long).mark_boxplot(color='lightgrey').encode(
            x=alt.X('n_classifiers:N', axis=alt.Axis(labelAngle=0)), 
            y='score:Q'
        )

        agg_df = df_long.groupby('n_classifiers').agg(['mean','std']).reset_index()
        agg_df.columns = ['n_classifiers','mean','std']
        agg_df['lower'] = agg_df.apply(lambda x: x['mean']-x['std'], axis=1)
        agg_df['upper'] = agg_df.apply(lambda x: x['mean']+x['std'], axis=1)

        # Create the confidence band
        area = alt.Chart(agg_df).mark_area(opacity=0.2, color='grey').encode(
            x=alt.X('n_classifiers:N', axis=alt.Axis(labelAngle=0, values=list(range(2,21,2)))),
            y=alt.Y('lower:Q', axis=alt.Axis(title=None)),
            y2='upper:Q',
        )

        # Create a DataFrame for the mean values
        best_df = pd.DataFrame({
            'n_classifiers': range(1, len(best_scores) + 1),
            'score': best_scores,
            'ensemble_solution': ensemble_solution,
            'ensemble_method': ensemble_method  
        })

        best_scores_df = pd.concat([best_scores_df, best_df])

        # Create the line chart using the mean values
        line = alt.Chart(best_df).mark_line(point=alt.OverlayMarkDef(color="black"), color='grey').encode(
            x=alt.X('n_classifiers:N', axis=alt.Axis(labelAngle=0)),
            y=alt.Y('score:Q', scale=alt.Scale(domain=[0, 1]), axis=alt.Axis(values=list(np.arange(0,1.2,0.2))) ),
        )

        # Combine the two charts
        combined = alt.layer(area, line).properties(
            title=alt.TitleParams(
                text=ensemble_solution,
                fontSize=fontsize,        # title size
                #font="Arial",
                #anchor="start"      # left-align
            )
            #title=ensemble_solution,
        )

        
        box_plots[ensemble_solution] = combined
    all_plots[ensemble_method] = box_plots


In [19]:
# Create a horizontal concatenation of all box plots for each ensemble method
box_plots = {}
for ensemble_method, plots in all_plots.items():
    box_plots[ensemble_method] = alt.hconcat(*plots.values()).properties(
        title=alt.TitleParams(
                text=f"{ensemble_method}",
                fontSize=fontsize,        # title size
                #font="Arial",
                #anchor="start"      # left-align
            )
        #title=f"Ensemble Method: {ensemble_method}",
    )

# Concatenate all ensemble method plots horizontally
final_plot = alt.vconcat(*box_plots.values()).configure_axis(
    labelFontSize=fontsize,
    titleFontSize=fontsize
)
final_plot 

alt.VConcatChart(...)

## Heterogeneous vs Homogeneous base classifiers

In [23]:
import ast 

ensemble_selection_homogeneous = pd.read_excel("outputs/multi_objective_validation/ensemble_selection_benbow_homogeneous.xlsx")
ensemble_selection_homogeneous['upper'] = ensemble_selection_homogeneous['best_fold_scores'].apply(lambda x: max(ast.literal_eval(x)))
ensemble_selection_homogeneous['lower'] = ensemble_selection_homogeneous['best_fold_scores'].apply(lambda x: min(ast.literal_eval(x)))
ensemble_selection_homogeneous['best_fold_scores'] = ensemble_selection_homogeneous['best_fold_scores'].apply(lambda x: ast.literal_eval(x))
ensemble_selection_homogeneous['n_classifier'] = ensemble_selection_homogeneous['candidates'].apply(lambda x: len(ast.literal_eval(x)))
ensemble_selection_homogeneous_exploded = ensemble_selection_homogeneous.explode('best_fold_scores')
best_ensemble_selection_homogeneous = ensemble_selection_homogeneous.groupby(['ensemble_method','solution','clf']).apply(pd.DataFrame.nlargest, n=1, columns=['best_mean_score'])[['candidates','n_classifier','best_mean_score','best_fold_scores']].reset_index()
best_ensemble_selection_homogeneous_exploded = best_ensemble_selection_homogeneous.explode('best_fold_scores')

/var/folders/ws/qg8ztb9n7yj1s15m1x9c1vdw0000gp/T/ipykernel_14039/1072722433.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  best_ensemble_selection_homogeneous = ensemble_selection_homogeneous.groupby(['ensemble_method','solution','clf']).apply(pd.DataFrame.nlargest, n=1, columns=['best_mean_score'])[['candidates','n_classifier','best_mean_score','best_fold_scores']].reset_index()


In [24]:
ensemble_selection_mix = pd.read_excel("outputs/multi_objective_validation/ensemble_selection_benbow_heterogeneous.xlsx")
ensemble_selection_mix['upper'] = ensemble_selection_mix['best_fold_scores'].apply(lambda x: max(ast.literal_eval(x)))
ensemble_selection_mix['lower'] = ensemble_selection_mix['best_fold_scores'].apply(lambda x: min(ast.literal_eval(x)))
ensemble_selection_mix['best_fold_scores'] = ensemble_selection_mix['best_fold_scores'].apply(lambda x: ast.literal_eval(x))
ensemble_selection_mix['n_classifier'] = ensemble_selection_mix['candidates'].apply(lambda x: len(ast.literal_eval(x)))
ensemble_selection_mix_exploded = ensemble_selection_mix.explode('best_fold_scores')
ensemble_selection_mix_exploded['clf'] = 'mixed'
best_ensemble_selection_mix = ensemble_selection_mix.groupby(['ensemble_method','solution']).apply(pd.DataFrame.nlargest, n=1, columns=['best_mean_score'])[['candidates','n_classifier','best_mean_score','best_fold_scores']].reset_index()
best_ensemble_selection_mix = best_ensemble_selection_mix.groupby(['ensemble_method','solution']).apply(pd.DataFrame.nlargest, n=1, columns=['best_mean_score'])[['candidates','n_classifier','best_mean_score','best_fold_scores']].reset_index()
best_ensemble_selection_mix_exploded = best_ensemble_selection_mix.explode('best_fold_scores')
best_ensemble_selection_mix_exploded['clf'] = 'heterogeneous'

best_ensemble_selection = pd.concat([best_ensemble_selection_homogeneous_exploded, best_ensemble_selection_mix_exploded])

/var/folders/ws/qg8ztb9n7yj1s15m1x9c1vdw0000gp/T/ipykernel_14039/3817540657.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  best_ensemble_selection_mix = ensemble_selection_mix.groupby(['ensemble_method','solution']).apply(pd.DataFrame.nlargest, n=1, columns=['best_mean_score'])[['candidates','n_classifier','best_mean_score','best_fold_scores']].reset_index()
/var/folders/ws/qg8ztb9n7yj1s15m1x9c1vdw0000gp/T/ipykernel_14039/3817540657.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explic

In [25]:
fontsize = 16
# draw boxplot of the ensemble and pairwise results
ensemble_comparison_chart = alt.Chart(best_ensemble_selection).mark_boxplot(extent='min-max', color='grey').encode(
    x=alt.X('clf:N', sort=['NaiveBayes','LogisticRegression','DecisionTree','RandomForest','SVM','mixed'], axis=alt.Axis(title=None, labelAngle=45)),
    y=alt.Y('best_fold_scores:Q', axis=alt.Axis(title=None)),
    #color=alt.Color('ensemble_type:N', title='Type'),
).properties(
    width=300,
    height=300,
).facet(
    column=alt.Column('ensemble_method:N', header=alt.Header(title='Ensemble Method', labelFontSize=fontsize, titleFontSize=fontsize)),
    row=alt.Row('solution:N', sort=['pfront','chull','best'], header=alt.Header(title='Ensemble Solution', labelFontSize=fontsize, titleFontSize=fontsize))
).configure_axis(
    labelFontSize=fontsize,
    titleFontSize=fontsize
)

ensemble_comparison_chart

alt.FacetChart(...)

## Run train_model.py

In [26]:
# python train_model.py

## Evaluate ensemble model vs single model

In [28]:
from Bio import SeqIO
import numpy as np
import pickle 
import os
import json
from utils.ensemble_crossval import get_classifiers
from sklearn.metrics import matthews_corrcoef, accuracy_score, precision_score, f1_score, fbeta_score, recall_score
from utils.util import true_positive, true_negative, false_positive, false_negative
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

def get_ensemble_model(candidates, ensemble_type='stacking'):
    # get list of candidate models
    models = get_classifiers(candidates)

    meta_models = {
            "stacking": StackingClassifier(estimators=None, final_estimator=LogisticRegression()),
            "voting_soft": VotingClassifier(estimators=None, voting="soft"),
            "voting_hard": VotingClassifier(estimators=None, voting="hard")
        }
    ensemble = meta_models[ensemble_type]
    ensemble.estimators = models
    return ensemble

def compute_eval_score(y_test, y_pred):
        eval_scores = {
                'Accuracy': accuracy_score(y_test, y_pred),
                'Precision': precision_score(y_test, y_pred),
                'Recall': recall_score(y_test, y_pred),
                'F_1': f1_score(y_test, y_pred), 
                'F_beta_0.5': fbeta_score(y_test, y_pred, beta=0.5),
                'F_beta_2': fbeta_score(y_test, y_pred, beta=2),
                'MCC': matthews_corrcoef(y_test, y_pred),
                'TP': true_positive(y_test, y_pred),
                'TN': true_negative(y_test, y_pred),
                'FP': false_positive(y_test, y_pred),
                'FN': false_negative(y_test, y_pred),
        }
        return eval_scores

In [29]:
#read train data
train_filename = "dataset/train_data/benbow.fasta"

X = []
y = []
groups = []
for record in SeqIO.parse(train_filename, "fasta"):
    X.append(str(record.seq))
    y.append(record.id.split('|')[1])
    groups.append(record.id.split('|')[0])

X_train = np.array(X)
y_train = np.array(y).astype(int)
groups_train = np.array(groups)

In [31]:
#read test data
test_filename = "dataset/test_data/benbow_test_data.fasta" 
X_test = []
y_test = []
groups_test = []
genera_test = []
species_test = []
desc_test = []
for record in SeqIO.parse(test_filename, "fasta"):
    X_test.append(str(record.seq))
    y_test.append(record.id.split('|')[1])
    groups_test.append(record.id.split('|')[0])
    genera_test.append(record.description.split(',')[0].split(' ')[2])
    species_test.append(' '.join(record.description.split(',')[0].split(' ')[2:]))
    desc_test.append('_'.join(record.id.split('|')[0].split('_')[0:2]) + ': ' + ' '.join(record.description.split(',')[0].split(' ')[1:]))


X_test = np.array(X_test).astype(str)
y_test = np.array(y_test).astype(int)
groups_test = np.array(groups_test)
genera_test = np.array(genera_test)
species_test = np.array(species_test)
desc_test = np.array(desc_test)

In [32]:
groups_train = np.array([group.split('.')[0] for group in groups_train])
groups_test = np.array([group.split('.')[0] for group in groups_test])

# ensure training data does not contain test data
test_ids = set(groups_test)

selected_ids = []
for i, g in enumerate(groups_train):
    if g not in test_ids:
        selected_ids.append(i)

reduced_X_train = X_train[selected_ids]
reduced_y_train = y_train[selected_ids]
reduced_group_train = groups_train[selected_ids]

print('X_train: ',reduced_X_train.shape)
print('X_test: ',X_test.shape)

#ensure species in train and test data sets are mutually exclusive
assert len(set(groups_test)-set(reduced_group_train)) == len(set(groups_test))

X_train:  (2569,)
X_test:  (566,)


In [34]:
import pickle 

#load model
model_path = "utils/models"
with open(os.path.join(model_path,'ensemble_stacking_benbow.pkl'), "rb") as input_file:
    ensemble_model = pickle.load(input_file)

with open(os.path.join(model_path,'RCKmer-7_SVM_benbow.pkl'), "rb") as input_file:
    single_model = pickle.load(input_file)

In [35]:
#compute evaluation scores
y_pred_ensemble = ensemble_model.predict(X_test)
y_prob_ensemble = ensemble_model.predict_proba(X_test)
ensemble_score = compute_eval_score(y_test, y_pred_ensemble)

y_pred_single = single_model.predict(X_test)
y_prob_single = single_model.predict_proba(X_test)
single_score = compute_eval_score(y_test, y_pred_single)

In [36]:
# Example dictionary
cm_dict = {
    'Ensemble': ensemble_score,
    'Single': single_score
}

# Convert to long-format DataFrame for Altair
rows = []
for model, counts in cm_dict.items():
    # Confusion matrix: rows=True label, columns=Predicted label
    cm_matrix = pd.DataFrame([[counts['TP'], counts['FN']],
                              [counts['FP'], counts['TN']]],
                             index=['1', '0'],
                             columns=['1', '0'])
    cm_long = cm_matrix.reset_index().melt(id_vars='index')
    cm_long.columns = ['Actual', 'Predicted', 'Count']
    cm_long['Model'] = model
    rows.append(cm_long)

df_cm = pd.concat(rows, ignore_index=True)

charts = {}
for model in cm_dict.keys():
    sub_df = df_cm[df_cm['Model'] == model]
    # Altair heatmap
    heatmap = alt.Chart(sub_df).mark_rect().encode(
        x=alt.X('Predicted:N', title='Predicted', axis=alt.Axis(labelAngle=0)),
        y=alt.Y('Actual:N', title='Actual'),
        color=alt.Color('Count:Q', scale=alt.Scale(scheme='greys'), legend=None),
    ).properties(
        width=200, 
        height=200,
        title=alt.TitleParams(model, anchor='end', orient='top', angle=0)  # Add Left-side title
    )

    # Add text labels
    text = alt.Chart(sub_df).mark_text(color='black').encode(
        x='Predicted:N',
        y='Actual:N',
        text='Count:Q',
        color=alt.condition(
        alt.datum.Count > 200,   # condition
        alt.value('white'),  # if True
        alt.value('black')  # if False
    )
    )

    chart = (heatmap + text)
    charts[model] = chart

cm_chart = alt.hconcat(*charts.values())
cm_chart

alt.HConcatChart(...)

## Compute evaluation metrics per accession number

In [37]:
# Define a function to apply per group
def compute_metrics(group_df):
    y_test = group_df['y_test'].values.astype(int)
    y_pred = group_df['y_pred'].values.astype(int)
    n_positive = np.sum(y_test)
    n_negative = len(y_test) - n_positive

    return pd.Series({
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F_1': f1_score(y_test, y_pred), 
        'F_beta_0.5': fbeta_score(y_test, y_pred, beta=0.5),
        'F_beta_2': fbeta_score(y_test, y_pred, beta=2),
        'MCC': matthews_corrcoef(y_test, y_pred),
        'positive': n_positive,
        'negative': n_negative,
    })


# read acc2taxid
tax_id_df = pd.read_excel('dataset/prev_studies/islandviewer/islandviewer_gis_taxid.xlsx')

predictions_df = pd.DataFrame([y_test, y_pred_ensemble, y_prob_ensemble, groups_test]).T 
predictions_df.columns = ['y_test', 'y_pred', 'y_prob', 'accession']
#assign genera and species from tax_id_df
predictions_df = predictions_df.merge(tax_id_df, on='accession', how='left')

single_predictions_df = pd.DataFrame([y_test, y_pred_single, y_prob_single, groups_test]).T 
single_predictions_df.columns = ['y_test', 'y_pred', 'y_prob', 'accession']
#assign genera and species from tax_id_df
single_predictions_df = single_predictions_df.merge(tax_id_df, on='accession', how='left')

group_by = 'accession_version'  # 'scientific_name' or 'genus' if you want to group by genus
group_metrics = predictions_df.groupby(group_by).apply(compute_metrics).reset_index()
single_group_metrics = single_predictions_df.groupby(group_by).apply(compute_metrics).reset_index()

group_metrics['model'] = 'ensemble'  
single_group_metrics['model'] = 'single'
merged_group_metrics = pd.concat([group_metrics, single_group_metrics], ignore_index=True)
#merged_group_metrics = merged_group_metrics[merged_group_metrics[group_by].str.contains('Escherichia')]  

label_order = merged_group_metrics[merged_group_metrics['model']=='single'].sort_values(by='F_1', ascending=False).accession_version.values
label_order = list(label_order)

/var/folders/ws/qg8ztb9n7yj1s15m1x9c1vdw0000gp/T/ipykernel_14039/920939432.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  group_metrics = predictions_df.groupby(group_by).apply(compute_metrics).reset_index()
/var/folders/ws/qg8ztb9n7yj1s15m1x9c1vdw0000gp/T/ipykernel_14039/920939432.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  single_group_metrics = single_predictions_df.groupby(group_by).apply(compute_m

In [39]:
fontsize = 18

#plot horizontal bar chart per genus
bar_chart = alt.Chart(merged_group_metrics).mark_bar(stroke='black', size=12).encode(
    y=alt.Y(f'{group_by}:N', 
            title='Accession Number', 
            sort=label_order, 
            axis=alt.Axis(labelAngle=0, orient='left', ),
            ),
    x=alt.X('F_1:Q', title='F1 Score', axis=alt.Axis(tickCount=10)),
    color=alt.Color('model:N',
                    scale=alt.Scale(
                        #scheme=['#000000','#ffffff']
                        domain=["ensemble", "single"],   # order of categories
                        range=['darkgrey','#f5f5f5']  # corresponding colors
                    ),
                    legend=alt.Legend(
                        orient='none',       # Disable default orientation
                        legendX=680,         # X-position of legend inside chart
                        legendY=730,           # Y-position of legend inside chart
                        titleFontSize=fontsize,    # legend title font size
                        labelFontSize=fontsize     # legend labels font size
                    )),
    #order=alt.Order('dataset:N', sort='descending')
    yOffset='model:N'
).properties(
    width=800, 
    height=800
)

# Melt to long format
df_long = group_metrics.melt(id_vars=f"{group_by}", value_vars=["positive", "negative"], var_name="class", value_name="n_samples")

# Compute proportions per row (normalize within each 'a')
df_long["proportion"] = df_long.groupby("class")["n_samples"].transform(lambda x: x / x.sum())
# Make negative values actually negative (so they plot on the left)
df_long.loc[df_long["class"] == "negative", "n_samples"] *= -1
# Add absolute value column for labeling
df_long["label"] = df_long["n_samples"].abs()

# Stacked bar chart
chart = alt.Chart(df_long).mark_bar().encode(
    y=alt.Y(f"{group_by}:N", 
            sort=label_order,
            #axis=alt.Axis(labels=True, title=None)),  # remove labels & title
            axis=None),
    x=alt.X("n_samples:Q", title="n_samples", axis=alt.Axis(labels=False, grid=False)),
    color=alt.Color("class:N",
                    scale=alt.Scale(
                        #scheme=['#1b9e77','#e7298a']
                        domain=["negative", "positive"],   # order of categories
                        range=['#66c2a5','#fc8d62']  # corresponding colors
                    ),
                    legend=alt.Legend(
                        orient='none',       # Disable default orientation
                        legendX=305,         # X-position of legend inside chart
                        legendY=730,           # Y-position of legend inside chart
                        titleFontSize=fontsize,    # legend title font size
                        labelFontSize=fontsize     # legend labels font size
                    )
                    )
).properties(width=400, height=800,)

# Labels for positive values (on the right side)
text_pos = alt.Chart(df_long[df_long["n_samples"] > 0]).mark_text(
    align="left", dx=3, fontSize=16
).encode(
    y=alt.Y(f"{group_by}:N", 
            sort=label_order,
            #axis=alt.Axis(labels=True, title=None)),  # remove labels & title
            axis=None),
    x="n_samples:Q",
    text="label:Q"
)

# Labels for negative values (on the left side)
text_neg = alt.Chart(df_long[df_long["n_samples"] < 0]).mark_text(
    align="right", dx=-3, fontSize=fontsize
).encode(
    y=alt.Y(f"{group_by}:N", 
            sort=label_order,
            #axis=alt.Axis(labels=True, title=None)),  # remove labels & title
            axis=None),
    x="n_samples:Q",
    text="label:Q"
)

samples_info = chart + text_pos + text_neg

layered_chart = alt.hconcat(bar_chart, samples_info).resolve_scale(
    y='shared',
    color='independent'
).configure_axis(
    labelFontSize=fontsize,
    titleFontSize=fontsize,
).configure_header(
    labelFontSize=fontsize,
    titleFontSize=fontsize,
)
layered_chart

alt.HConcatChart(...)

In [42]:
# layered_chart.save("figures/performance_per_species_benbow.pdf")

## Evaluation for boundaries prediction task

In [43]:
def prepare_data(json_file, type='test'):
    from utils.util import read_eval_result
    eval_df = read_eval_result(json_file)
    stats_eval_df = eval_df.groupby('Predictor').describe().reset_index()
    stats_eval_df = stats_eval_df[stats_eval_df['Predictor'].str.startswith('RCKmer') | stats_eval_df['Predictor'].str.startswith('ensemble')]
    metrics = ['MCC', 'F-Score', 'F-2-Score', 'Precision', 'Recall'] 
    predictors = stats_eval_df['Predictor'].unique()

    data = []
    for metric in metrics:
        for predictor in predictors:
            sub_data = stats_eval_df[stats_eval_df['Predictor'] == predictor][metric]
            model = '_'.join(predictor.split('_')[0:-2])
            window_size = int(predictor.split('_')[-2])
            threshold = float(predictor.split('_')[-1])

            mean = sub_data['mean'].values[0]
            std = sub_data['std'].values[0]

            data.append({
                'model': model,
                'window_size': window_size,
                'threshold': threshold,
                'metric': metric,
                'mean': mean,
                'std': std,
                'lower': mean - std,
                'upper': mean + std
            })

    df = pd.DataFrame(data)
    df['dataset'] = type
    return df 

test_json_file = "outputs/boundaries_predictions/evaluation/evaluation_result_test_gridsearch_latest.json"
#literature_json_file = "outputs/boundaries_predictions/evaluation/evaluation_result_literature_gridsearch_latest.json"

df = prepare_data(test_json_file, type='benbow_test')
#literature_eval_df = prepare_data(literature_json_file, type='literature')


In [ ]:
#plot charts for visual comparison
#select models to compare
models = ['RCKmer-7_SVM_fine_tuned_benbow_test',
          'RCKmer-7_SVM_Subsequence_RF_stacking_fine_tuned_benbow_test'] #RCKmer-7_SVM_Subsequence_RF_stacking_fine_tuned_benbow_test, ensemble_voting_soft_benbow
thresholds = [0.8]
metrics = ['F-Score']
datasets = ['benbow_test']

charts = {}
fontsize = 16

for threshold in [0.55,0.6,0.65,0.7,0.75,0.8]: 
    #filter df
    selected_df = df[(df['model'].isin(models)) & (df['threshold']==threshold) & (df['metric'].isin(metrics) & (df['dataset'].isin(datasets)))]
    #rename the models to be single or ensemble model
    selected_df['model'] = selected_df['model'].apply(lambda x: 'Ensemble' if 'stacking' in x else 'Single')


    # Create the confidence band
    selected_band = alt.Chart(selected_df).mark_area(opacity=0.2).encode(
        x=alt.X('window_size:O', axis=alt.Axis(labelAngle=0, title='Window Size')),
        y='lower',
        y2='upper',
        color=alt.Color('model:N', title='Model'),
        tooltip=['model', 'window_size', 'threshold', 'mean', 'lower', 'upper']
    )

    # Create the main line
    selected_line = alt.Chart(selected_df).mark_line(point=True).encode(
        x=alt.X('window_size:O', axis=alt.Axis(labelAngle=0, title='Window Size', values=list(range(5000,17000,2000)))),
        y=alt.Y('mean:Q', axis=alt.Axis(title='F1 score')),
        color=alt.Color('model:N', title='Model'),
        tooltip=['model', 'window_size', 'threshold', 'mean', 'lower', 'upper']
    )

    # Combine and show
    selected_chart = (selected_band + selected_line).properties( 
        width=500,
        height=300,
        #title=f"Threshold = {threshold}",
        title=alt.TitleParams(
                text=f"Threshold = {threshold}",
                fontSize=fontsize,        # title size
            )
    )

    charts[threshold] = selected_chart

first_items = list(charts.items())[:2]
second_items = list(charts.items())[2:4]
third_items = list(charts.items())[4:]
all_charts = alt.vconcat(
    alt.hconcat(*dict(first_items).values()),
    alt.hconcat(*dict(second_items).values()),
    alt.hconcat(*dict(third_items).values())
).configure_axis(
    labelFontSize=fontsize,
    titleFontSize=fontsize
)
all_charts


/var/folders/ws/qg8ztb9n7yj1s15m1x9c1vdw0000gp/T/ipykernel_14039/2101879248.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_df['model'] = selected_df['model'].apply(lambda x: 'Ensemble' if 'stacking' in x else 'Single')


alt.VConcatChart(...)

In [45]:
#plot charts for visual comparison
#select models to compare
models = ['RCKmer-7_SVM_fine_tuned_benbow_test',
          'ensemble_voting_soft_benbow'] #RCKmer-7_SVM_Subsequence_RF_stacking_fine_tuned_benbow_test, ensemble_voting_soft_benbow
thresholds = [0.8]
metrics = ['F-Score']
datasets = ['benbow_test']

charts = {}
fontsize = 16

for threshold in [0.55,0.6,0.65,0.7,0.75,0.8]: 
    #filter df
    selected_df = df[(df['model'].isin(models)) & (df['threshold']==threshold) & (df['metric'].isin(metrics) & (df['dataset'].isin(datasets)))]
    #rename the models to be single or ensemble model
    selected_df['model'] = selected_df['model'].apply(lambda x: 'Ensemble' if 'ensemble' in x else 'Single')


    # Create the confidence band
    selected_band = alt.Chart(selected_df).mark_area(opacity=0.2).encode(
        x=alt.X('window_size:O', axis=alt.Axis(labelAngle=0, title='Window Size')),
        y='lower',
        y2='upper',
        color=alt.Color('model:N', title='Model'),
        tooltip=['model', 'window_size', 'threshold', 'mean', 'lower', 'upper']
    )

    # Create the main line
    selected_line = alt.Chart(selected_df).mark_line(point=True).encode(
        x=alt.X('window_size:O', axis=alt.Axis(labelAngle=0, title='Window Size', values=list(range(5000,17000,2000)))),
        y=alt.Y('mean:Q', axis=alt.Axis(title='F1 score')),
        color=alt.Color('model:N', title='Model'),
        tooltip=['model', 'window_size', 'threshold', 'mean', 'lower', 'upper']
    )

    # Combine and show
    selected_chart = (selected_band + selected_line).properties( 
        width=500,
        height=300,
        #title=f"Threshold = {threshold}",
        title=alt.TitleParams(
                text=f"Threshold = {threshold}",
                fontSize=fontsize,        # title size
                #font="Arial",
                #anchor="start"      # left-align
            )
    )

    charts[threshold] = selected_chart

first_items = list(charts.items())[:2]
second_items = list(charts.items())[2:4]
third_items = list(charts.items())[4:]
all_charts = alt.vconcat(
    alt.hconcat(*dict(first_items).values()),
    alt.hconcat(*dict(second_items).values()),
    alt.hconcat(*dict(third_items).values())
).configure_axis(
    labelFontSize=fontsize,
    titleFontSize=fontsize
)
all_charts


/var/folders/ws/qg8ztb9n7yj1s15m1x9c1vdw0000gp/T/ipykernel_14039/845749636.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_df['model'] = selected_df['model'].apply(lambda x: 'Ensemble' if 'ensemble' in x else 'Single')


alt.VConcatChart(...)